# Autograd — Automatic Differentiation

Autograd is the engine behind all neural network training in PyTorch.
It automatically computes gradients for you — no manual calculus needed.

**Why it matters:** Training a neural network = find weights that minimize loss.
To minimize, you need the gradient: "if I change this weight slightly, how does the loss change?"
Autograd computes this for every weight automatically via backpropagation.

In [ ]:
import torch

# requires_grad=True — tells PyTorch to track operations on this tensor
# so it can compute gradients later
x = torch.tensor(3.0, requires_grad=True)

print('x:             ', x)
print('requires_grad: ', x.requires_grad)
print('grad_fn:       ', x.grad_fn)   # None — x is a leaf (we created it directly)

## Forward pass — building the computation graph

Every operation on a `requires_grad` tensor is recorded.
PyTorch builds a **computation graph** that tracks how each value was computed.
This is used to compute gradients during backward pass.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

# y = x^2 + 2x + 1
y = x**2 + 2*x + 1

print('x:        ', x.data)
print('y:        ', y.data)       # 9 + 6 + 1 = 16
print('grad_fn:  ', y.grad_fn)    # AddBackward — knows how to backprop through addition

## Backward pass — computing gradients

`.backward()` traverses the computation graph in reverse and computes
d(loss)/d(weight) for every tensor with `requires_grad=True`.

For y = x² + 2x + 1:
- dy/dx = 2x + 2
- At x=3: dy/dx = 2(3) + 2 = **8**

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1

y.backward()   # compute dy/dx

print('gradient dy/dx at x=3:', x.grad)   # should be 8.0
# dy/dx = 2x + 2 = 2(3) + 2 = 8

## Multi-variable gradients — how a linear layer works

In a real neural network: `loss = f(W, b)` where W and b are weights.
We need d(loss)/dW and d(loss)/db to update them.

In [ ]:
# Simulate one step of a linear layer: y = Wx + b, loss = (y - target)^2

x      = torch.tensor([1.0, 2.0, 3.0])           # input (fixed, no grad)
W      = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)   # weights
b      = torch.tensor(0.0, requires_grad=True)               # bias
target = torch.tensor(5.0)                        # what we want y to be

# Forward pass
y    = (W * x).sum() + b     # dot product + bias = 0.5+1.0+1.5 + 0 = 3.0
loss = (y - target)**2       # MSE loss = (3.0 - 5.0)^2 = 4.0

print('y:    ', y.item())
print('loss: ', loss.item())

# Backward pass
loss.backward()

print('dL/dW:', W.grad)   # gradient of loss w.r.t. each weight
print('dL/db:', b.grad)   # gradient of loss w.r.t. bias

## CRITICAL: zero_grad() — the most common training bug

Gradients **accumulate** by default. Every `.backward()` call ADDS to `.grad`.
You must zero them out before each backward pass.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# WITHOUT zeroing — gradients accumulate (WRONG for training)
for i in range(3):
    y = x**2
    y.backward()
    print(f'step {i+1} grad (accumulating): {x.grad}')  # 4, 8, 12 — WRONG

print()

# Manually zero the grad
x.grad = None

# WITH zeroing — each step is independent (CORRECT)
for i in range(3):
    y = x**2
    y.backward()
    print(f'step {i+1} grad (correct):       {x.grad}')  # 4, 4, 4 — correct
    x.grad.zero_()   # zero out after reading

## torch.no_grad() — disable gradient tracking

During inference (prediction), you don't need gradients.
`no_grad()` skips building the computation graph → faster + less memory.

Always use this during evaluation and inference.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

# With gradients (training mode)
y = x**2
print('has grad_fn:', y.grad_fn is not None)   # True

# Without gradients (inference mode)
with torch.no_grad():
    y = x**2
    print('has grad_fn:', y.grad_fn is not None)  # False — no graph built
    print('y:', y)                                  # still computes the value

# Decorator version — useful for eval functions
@torch.no_grad()
def predict(model_input):
    return model_input ** 2

result = predict(x)
print('result grad_fn:', result.grad_fn)   # None

## The full picture — what happens in every training step

```
1. Forward pass:    output = model(input)          → builds computation graph
2. Compute loss:    loss = loss_fn(output, target)
3. Zero grads:      optimizer.zero_grad()          → clear previous gradients
4. Backward pass:   loss.backward()                → compute all gradients
5. Update weights:  optimizer.step()               → W = W - lr * W.grad
```

This 5-step loop is ALL of deep learning training. Everything else is details.

In [ ]:
# Manual gradient descent — without an optimizer
# Shows exactly what happens under the hood

torch.manual_seed(42)

W      = torch.tensor([0.1, 0.1, 0.1], requires_grad=True)
b      = torch.tensor(0.0, requires_grad=True)
x      = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor(5.0)
lr     = 0.01

for step in range(20):
    # 1. Forward
    y    = (W * x).sum() + b
    loss = (y - target)**2

    # 2. Zero grads
    if W.grad is not None:
        W.grad.zero_()
        b.grad.zero_()

    # 3. Backward
    loss.backward()

    # 4. Update weights manually (this is what optimizer.step() does)
    with torch.no_grad():   # no_grad because we don't want to track these updates
        W -= lr * W.grad
        b -= lr * b.grad

    if step % 5 == 0:
        print(f'step {step:2d} | loss: {loss.item():.4f} | y: {y.item():.4f}')